# (Opsiyonal) Olist reviews' - Translations...

* 🇧🇷 Brezilya Portekizcesi bilmiyorsanız, hadi `order_reviews` metinlerini 🇬🇧 İngilizce’ye çevirelim!

* Bunun için `pip install googletrans==4.0.2` yüklemeniz gerekecek.

☢️ Bu API ile herhangi bir sorun yaşarsanız, bunu düzeltmek için zaman harcamayın, bu package oldukça dengesiz… Aklınızda bulunsun:
- bu optional bir challenge
- Brezilya Portekizcesi ile yazılmış review’ların anlamını görmek için bazı yorumları Google Translate’e kopyalayıp yapıştırarak yine de eski yöntemle yapabilirsiniz.

In [1]:
import sys, inspect
import googletrans
from googletrans import Translator

print("python:", sys.executable)
print("googletrans version:", googletrans.__version__)
print("translate is coroutine?", inspect.iscoroutinefunction(Translator.translate))

python: /Users/gizemtotkanli/.pyenv/versions/3.12.9/envs/workintech_current/bin/python
googletrans version: 3.4.0
translate is coroutine? True


In [2]:
import googletrans, inspect
from googletrans import Translator

print("googletrans version:", googletrans.__version__)
print("Translator.translate is coroutine?:", inspect.iscoroutinefunction(Translator.translate))

googletrans version: 3.4.0
Translator.translate is coroutine?: True


## Review Translator

👉 `reviews` dataset’ini load edin ve 1-yıldız rating’e sahip review’lardan bir örnek (sample) seçin.

In [3]:
from olist.data import Olist
data = Olist().get_data()

👀 20 adet düşük puan alan yorumdan oluşan bir örneklem seçin (rastgele) ve bunu bir listeye dönüştürün.

In [4]:
import pandas as pd

reviews = data["order_reviews"].copy()

# 1 yıldız + boş olmayan yorumlar
one_star = reviews.loc[
    (reviews["review_score"] == 1) &
    (reviews["review_comment_message"].notna()) &
    (reviews["review_comment_message"].astype(str).str.strip().ne(""))
].copy()

# 20 örnek seç (tekrarlanabilir olsun)
sample_reviews = one_star.sample(n=20, random_state=42)

texts = sample_reviews["review_comment_message"].astype(str).tolist()

len(texts), texts[0][:120]

(20, 'Demora de entrega')

🗣 Bu göreve başlamadan önce önceden yüklediğiniz `google_translator` paketini kullanarak bu yorumları çevirin:

In [5]:
import asyncio
import inspect
import time
from googletrans import Translator

translator = Translator()

async def translate_one(t: str) -> str:
    """Tek metni güvenle çevirir; async/sync farkını otomatik yönetir."""
    if not isinstance(t, str) or not t.strip():
        return ""
    try:
        if inspect.iscoroutinefunction(translator.translate):
            res = await translator.translate(t, src="pt", dest="en")
        else:
            res = translator.translate(t, src="pt", dest="en")
        return getattr(res, "text", str(res))
    except Exception as e:
        # İstersen hata mesajını da döndürebilirsin
        return "TRANSLATION_FAILED"

async def translate_many(texts, sleep_s: float = 0.4):
    out = []
    for t in texts:
        out.append(await translate_one(t))
        # API'yı boğmamak için küçük bekleme
        await asyncio.sleep(sleep_s)
    return out

translated = await translate_many(texts, sleep_s=0.4)

len(translated), translated[0][:120]

(20, 'Delivery delay')

In [6]:
import pandas as pd

df_translated = pd.DataFrame({
    "original_text": texts,
    "translated_text": translated
})

df_translated.head(10)

,original_text,translated_text
0,Demora de entrega,Delivery delay
1,"Depois que fiz a compra, passou 30 dias para a...","After I made the purchase, it took 30 days for..."
2,"ainda não entregaram meu produto, apenas o out...","They still haven't delivered my product, just ..."
3,"Não estou satisfeita, pois ainda não recebi o ...",I'm not satisfied as I haven't received the pr...
4,Até hoje não recebi meu produto. Vou fazer uma...,To this day I have not received my product. I'...
5,O produto já passou da data de entrega e não c...,The product is past its delivery date and I ca...
6,"Até o momento não recebi o produto , não entra...","So far I have not received the product, they h..."
7,Produto veio faltando duas peças e eu pedi a t...,Product arrived missing two pieces and I asked...
8,comprei dois produtos há mais de um mês e não ...,I bought two products over a month ago and I h...
9,Solicitei informações acerca da entrega de meu...,I requested information about the delivery of ...


**Insights** 💡
- Bazı kötü review’lar delivery ile ilgili (`wait_time`, kaçırılan teslim tarihi, vb.)
- Ancak bazı kötü review’lar seller veya ürünle ilgili...

Peki iki olası nedeni nasıl ayırt edebiliriz?

Bu Olist’in mutlaka bilmesi gereken bir şey:
- bazı ürünleri katalogdan mı çıkarmalı?
- yoksa bazı seller’ları marketplace’ten mi kaldırmalı?


In [7]:
# Basit tema ipuçları (keyword bazlı hızlı tarama)
keywords = {
    "delivery_delay": ["delay", "late", "delivery", "arrived", "waiting"],
    "missing_damaged": ["missing", "damage", "broken", "defect", "wrong product"],
    "seller_service": ["seller", "refund", "support", "response", "cancel"],
}

def tag_text(s):
    s_low = str(s).lower()
    tags = []
    for k, ws in keywords.items():
        if any(w in s_low for w in ws):
            tags.append(k)
    return tags

df_translated["tags"] = df_translated["translated_text"].apply(tag_text)
df_translated[["translated_text", "tags"]].head(10)

,translated_text,tags
0,Delivery delay,[delivery_delay]
1,"After I made the purchase, it took 30 days for...",[]
2,"They still haven't delivered my product, just ...",[]
3,I'm not satisfied as I haven't received the pr...,[]
4,To this day I have not received my product. I'...,[]
5,The product is past its delivery date and I ca...,[delivery_delay]
6,"So far I have not received the product, they h...",[]
7,Product arrived missing two pieces and I asked...,"[delivery_delay, missing_damaged, seller_service]"
8,I bought two products over a month ago and I h...,[]
9,I requested information about the delivery of ...,[delivery_delay]
